<a href="https://colab.research.google.com/github/Xuli2317/BOT_FPO_DATA/blob/main/FPO_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote
from config import *
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import random
import time

# FPO

In [ ]:
BASE_URL = "https://www.fpo.go.th/main/Statistic-Database.aspx"
SAVE_DIR = FPO_DIR
FPO_CHECKPOINT_DIR = os.path.join(FPO_CODE_DIR,"checkpoint")
FPO_CHECKPOINT_FILE = os.path.join(FPO_CHECKPOINT_DIR,"checkpoint.parquet")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(FPO_CHECKPOINT_DIR, exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0"}
time.sleep(random.uniform(2,5))
r = requests.get(BASE_URL, headers=headers, timeout=30)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

rows = []
current_category = None

In [ ]:
for tr in soup.select("tr"):
    cells = [td.get_text(" ", strip=True) for td in tr.select("td")]

    if not cells:
        continue

    row_text = " ".join(cells)

    if row_text.startswith("หมวด"):
        current_category = row_text
        continue

    code_match = re.search(r"\b[A-Za-z]+_[A-Za-z]?\d+\b|Macro_RSI", row_text)

    if not code_match:
        continue

    data_code = code_match.group(0)

    links = []

    for a in tr.select("a[href]"):
        href = urljoin(BASE_URL, a["href"])

        img_alt = ""
        img = a.find("img")

        if img:
            img_alt = img.get("alt", "").lower()

        link_type = None
        href_lower = href.lower()

        if "xls" in img_alt or ".xls" in href_lower:
            link_type = "xls"
        elif "pdf" in img_alt or ".pdf" in href_lower:
            link_type = "pdf"
        elif "csv" in img_alt or ".csv" in href_lower:
            link_type = "csv"

        if link_type:
            links.append({
                "type": link_type,
                "url": href
            })

    rows.append({
        "category": current_category,
        "data_code": data_code,
        "raw_text": row_text,
        "links": links
    })

In [ ]:
df_links = pd.DataFrame(rows)
df_files = df_links.explode("links").dropna(subset=["links"]).copy()
df_files["file_type"] = df_files["links"].apply(lambda x: x["type"])
df_files["url"] = df_files["links"].apply(lambda x: x["url"])
df_files = df_files.drop(columns=["links"])
df_files["file_key"] = (
    df_files["data_code"].astype(str)
    + "_"
    + df_files["file_type"].astype(str)
)

df_files.to_excel(os.path.join(SAVE_DIR,"fpo_statistic_links.xlsx"),index=False)
df_files.head()

(77, 6)


,category,data_code,raw_text,file_type,url,file_key
0,หมวดเศรษฐกิจมหภาค,Macro_RSI,Macro_RSI ดัชนีความเชื่อมั่นอนาคตเศรษฐกิจภูมิภ...,xls,https://www.fpo.go.th/main/getattachment/6d6ce...,Macro_RSI_xls
0,หมวดเศรษฐกิจมหภาค,Macro_RSI,Macro_RSI ดัชนีความเชื่อมั่นอนาคตเศรษฐกิจภูมิภ...,pdf,https://www.fpo.go.th/main/getattachment/125a3...,Macro_RSI_pdf
1,หมวดการคลังและภาษีอากร,FIT_D101,FIT_D101 ผลการจัดเก็บรายได้รัฐบาล 29/5/2569 ...,xls,https://www.fpo.go.th/main/getattachment/b5a16...,FIT_D101_xls
1,หมวดการคลังและภาษีอากร,FIT_D101,FIT_D101 ผลการจัดเก็บรายได้รัฐบาล 29/5/2569 ...,pdf,https://www.fpo.go.th/main/getattachment/4dd85...,FIT_D101_pdf
2,หมวดการคลังและภาษีอากร,FIT_D104,FIT_D104 โครงสร้างงบประมาณ 30/4/2569 สายทิพย...,xls,https://www.fpo.go.th/main/getattachment/23950...,FIT_D104_xls


In [ ]:
if os.path.exists(FPO_CHECKPOINT_FILE):
    df_checkpoint = pd.read_parquet(FPO_CHECKPOINT_FILE)
    done_files = set(df_checkpoint["file_key"])

else:
    df_checkpoint = pd.DataFrame(columns=[
        "file_key",
        "category",
        "data_code",
        "file_type",
        "url",
        "path"
    ])
    done_files = set()
df_test = df_files.copy()
df_test = df_files[~df_files["file_key"].isin(done_files)].reset_index(drop=True)

print("Done:", len(done_files))
print("Remaining:", len(df_test))

Loaded checkpoint: 61


In [ ]:
FPO_MAX_WORKERS = 2
FPO_CHECKPOINT_EVERY = 10

fpo_lock = threading.Lock()
new_rows = []

In [ ]:
def download_fpo_file(row):
    file_key = row["file_key"]

    try:
        code = row["data_code"]
        file_type = row["file_type"]
        url = row["url"]

        folder = os.path.join(SAVE_DIR, code)
        os.makedirs(folder, exist_ok=True)

        time.sleep(random.uniform(2,5))

        r = requests.get(
            url,
            headers=headers,
            timeout=60
        )
        r.raise_for_status()

        path = os.path.join(
            folder,
            f"{code}.{file_type}"
        )

        with open(path, "wb") as f:
            f.write(r.content)

        return {
            "file_key": file_key,
            "category": row["category"],
            "data_code": code,
            "file_type": file_type,
            "url": url,
            "path": path
        }, None

    except Exception as e:
        return None, (file_key, e)

In [ ]:
def save_fpo_checkpoint():
    if not new_rows:
        return

    global df_checkpoint

    df_checkpoint = pd.concat(
        [df_checkpoint, pd.DataFrame(new_rows)],
        ignore_index=True
    )

    df_checkpoint = df_checkpoint.drop_duplicates(
        subset=["file_key"],
        keep="last"
    )

    df_checkpoint.to_parquet(
        FPO_CHECKPOINT_FILE,
        index=False
    )

In [ ]:
if not df_test.empty:
    with ThreadPoolExecutor(max_workers=FPO_MAX_WORKERS) as executor:
        futures = {
            executor.submit(download_fpo_file, row): row["file_key"]
            for _, row in df_test.iterrows()
        }

        for i, future in enumerate(as_completed(futures), start=1):
            file_key = futures[future]
            new_row, err = future.result()

            if new_row is not None:
                with fpo_lock:
                    new_rows.append(new_row)
                    done_files.add(file_key)

            if err is not None:
                _, e = err
                print(f"ERROR : {file_key}")
                print(e)

            if i % FPO_CHECKPOINT_EVERY == 0:
                with fpo_lock:
                    save_fpo_checkpoint()
                    new_rows.clear()
            if i % 10 == 0:
              sleep_time = random.uniform(15,30)
              time.sleep(sleep_time)

# final checkpoint after the loop finishes
with fpo_lock:
    save_fpo_checkpoint()
    new_rows.clear()